# Mask R-CNN + MobileNetV3-FPN untuk Deteksi dan Segmentasi Lubang Jalan

Notebook ini berisi pipeline lengkap sampai tahap **inferensi video**:

1. Setup library dan konfigurasi project  
2. Dataset loader COCO JSON untuk Mask R-CNN  
3. DataLoader train/validation  
4. Model Mask R-CNN + MobileNetV3-Large + Feature Pyramid Network (FPN)  
5. Training loop dan checkpoint  
6. Load best model  
7. Inferensi gambar  
8. Batch inferensi folder test  
9. Inferensi video  

## Struktur Dataset yang Diharapkan

```text
pothole_maskrcnn_mobilenet/
├── data/
│   ├── images/
│   │   ├── train/
│   │   ├── val/
│   │   └── test/
│   ├── annotations/
│   │   ├── train.json
│   │   ├── val.json
│   │   └── test.json
│   └── videos/
│       └── pothole_test.mp4
├── checkpoints/
├── outputs/
└── notebooks/
    └── 01_train_maskrcnn_mobilenetv3_fpn_video.ipynb
```

Anotasi disarankan memakai format **COCO JSON** dengan polygon mask.


## Cell 1 — Import Library dan Setup Device

In [ ]:
# ============================================================
# Cell 1 - Import Library and Setup Device
# ============================================================

import os
import csv
import json
import time
import random
import math
import yaml
from pathlib import Path
from datetime import datetime
from contextlib import nullcontext

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torchvision

from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as F
from torchvision.models.detection import MaskRCNN
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn

from pycocotools.coco import COCO
from tqdm.auto import tqdm

print("PyTorch version     :", torch.__version__)
print("TorchVision version :", torchvision.__version__)
print("CUDA available      :", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device              :", device)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


SEED = 42
set_seed(SEED)


## Cell 2 — Konfigurasi Project

In [ ]:
# ============================================================
# Cell 2 - Project Configuration from YAML
# ============================================================

# ------------------------------------------------------------
# 1. Cari folder project dan file YAML
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd()

# Jika notebook dijalankan dari root project
if (CURRENT_DIR / "configs" / "config_maskrcnn_mobilenetv3_fpn.yaml").exists():
    PROJECT_DIR = CURRENT_DIR

# Jika notebook dijalankan dari folder notebooks/
elif (CURRENT_DIR.parent / "configs" / "config_maskrcnn_mobilenetv3_fpn.yaml").exists():
    PROJECT_DIR = CURRENT_DIR.parent

else:
    raise FileNotFoundError(
        "File configs/config_maskrcnn_mobilenetv3_fpn.yaml tidak ditemukan. "
        "Pastikan notebook dijalankan dari root project atau dari folder notebooks."
    )

CONFIG_PATH = PROJECT_DIR / "configs" / "config_maskrcnn_mobilenetv3_fpn.yaml"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

print("Config loaded from:", CONFIG_PATH)

# ------------------------------------------------------------
# 2. Seed dan device
# ------------------------------------------------------------

SEED = CFG["project"]["seed"]
set_seed(SEED)

device_cfg = CFG["project"].get("device", "auto")

if device_cfg == "auto":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
elif device_cfg == "cuda":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    device = torch.device("cpu")

print("Device:", device)

# ------------------------------------------------------------
# 3. Path project
# ------------------------------------------------------------

DATA_DIR = PROJECT_DIR / CFG["paths"]["data_dir"]
IMAGE_DIR = PROJECT_DIR / CFG["paths"]["image_dir"]
ANNOTATION_DIR = PROJECT_DIR / CFG["paths"]["annotation_dir"]
VIDEO_DIR = PROJECT_DIR / CFG["paths"]["video_dir"]

TRAIN_IMG_DIR = PROJECT_DIR / CFG["paths"]["train_img_dir"]
VAL_IMG_DIR = PROJECT_DIR / CFG["paths"]["val_img_dir"]
TEST_IMG_DIR = PROJECT_DIR / CFG["paths"]["test_img_dir"]

TRAIN_JSON = PROJECT_DIR / CFG["paths"]["train_json"]
VAL_JSON = PROJECT_DIR / CFG["paths"]["val_json"]
TEST_JSON = PROJECT_DIR / CFG["paths"]["test_json"]

CHECKPOINT_DIR = PROJECT_DIR / CFG["paths"]["checkpoint_dir"]
OUTPUT_DIR = PROJECT_DIR / CFG["paths"]["output_dir"]
LOG_DIR = PROJECT_DIR / CFG["paths"]["log_dir"]
PRED_DIR = PROJECT_DIR / CFG["paths"]["prediction_dir"]
METRIC_DIR = PROJECT_DIR / CFG["paths"]["metric_dir"]

# ------------------------------------------------------------
# 4. Dataset
# ------------------------------------------------------------

CLASS_NAMES = CFG["dataset"]["class_names"]
NUM_CLASSES = CFG["dataset"]["num_classes"]
INPUT_SIZE = CFG["dataset"]["input_size"]

MAP_ALL_CATEGORIES_TO_SINGLE_CLASS = CFG["dataset"].get(
    "map_all_categories_to_single_class",
    True
)

# ------------------------------------------------------------
# 5. Augmentation
# ------------------------------------------------------------

AUG_HFLIP_ENABLED = CFG["augmentation"]["train"]["random_horizontal_flip"]["enabled"]
AUG_HFLIP_PROB = CFG["augmentation"]["train"]["random_horizontal_flip"]["probability"]

# ------------------------------------------------------------
# 6. Model
# ------------------------------------------------------------

MODEL_ARCHITECTURE = CFG["model"]["architecture"]
MODEL_BACKBONE = CFG["model"]["backbone"]

PRETRAINED_BACKBONE = CFG["model"]["pretrained_backbone"]
TRAINABLE_BACKBONE_LAYERS = CFG["model"]["trainable_backbone_layers"]

RPN_PRE_NMS_TOP_N_TRAIN = CFG["model"]["rpn_pre_nms_top_n_train"]
RPN_POST_NMS_TOP_N_TRAIN = CFG["model"]["rpn_post_nms_top_n_train"]
RPN_PRE_NMS_TOP_N_TEST = CFG["model"]["rpn_pre_nms_top_n_test"]
RPN_POST_NMS_TOP_N_TEST = CFG["model"]["rpn_post_nms_top_n_test"]

BOX_DETECTIONS_PER_IMG = CFG["model"]["box_detections_per_img"]
BOX_SCORE_THRESH = CFG["model"]["box_score_thresh"]

# ------------------------------------------------------------
# 7. Training hyperparameters
# ------------------------------------------------------------

NUM_EPOCHS = CFG["training"]["epochs"]
BATCH_SIZE = CFG["training"]["batch_size"]
VAL_BATCH_SIZE = CFG["training"]["val_batch_size"]

NUM_WORKERS = CFG["training"]["num_workers"]
PIN_MEMORY = CFG["training"]["pin_memory"]

OPTIMIZER_NAME = CFG["training"]["optimizer"]["name"]
LEARNING_RATE = CFG["training"]["optimizer"]["learning_rate"]
WEIGHT_DECAY = CFG["training"]["optimizer"]["weight_decay"]

SCHEDULER_NAME = CFG["training"]["scheduler"]["name"]
SCHEDULER_MODE = CFG["training"]["scheduler"]["mode"]
SCHEDULER_FACTOR = CFG["training"]["scheduler"]["factor"]
SCHEDULER_PATIENCE = CFG["training"]["scheduler"]["patience"]

USE_AMP_CONFIG = CFG["training"]["mixed_precision"]["enabled"]

GRAD_CLIP_ENABLED = CFG["training"]["gradient_clipping"]["enabled"]
GRAD_CLIP_NORM = CFG["training"]["gradient_clipping"]["max_norm"]

BEST_MODEL_NAME = CFG["training"]["checkpoint"]["best_model_name"]
LAST_MODEL_NAME = CFG["training"]["checkpoint"]["last_model_name"]

LOG_CSV_NAME = CFG["training"]["logging"]["csv_name"]
LOSS_PLOT_NAME = CFG["training"]["logging"]["loss_plot_name"]

# ------------------------------------------------------------
# 8. Inference
# ------------------------------------------------------------

SCORE_THRESHOLD = CFG["inference"]["score_threshold"]
MASK_THRESHOLD = CFG["inference"]["mask_threshold"]
MAX_TEST_IMAGES = CFG["inference"]["max_test_images"]
PREDICTION_IMAGE_SUFFIX = CFG["inference"]["prediction_image_suffix"]

# ------------------------------------------------------------
# 9. Video inference
# ------------------------------------------------------------

INPUT_VIDEO_NAME = CFG["video_inference"]["input_video_name"]
OUTPUT_VIDEO_NAME = CFG["video_inference"]["output_video_name"]
PROCESS_EVERY_N_FRAMES = CFG["video_inference"]["process_every_n_frames"]
MAX_VIDEO_FRAMES = CFG["video_inference"]["max_frames"]
VIDEO_CODEC = CFG["video_inference"]["codec"]
MASK_ALPHA = CFG["video_inference"]["mask_alpha"]

print("\nConfiguration summary:")
print("PROJECT_DIR                  :", PROJECT_DIR)
print("MODEL                        :", MODEL_ARCHITECTURE, "+", MODEL_BACKBONE)
print("INPUT_SIZE                   :", INPUT_SIZE)
print("NUM_CLASSES                  :", NUM_CLASSES)
print("NUM_EPOCHS                   :", NUM_EPOCHS)
print("BATCH_SIZE                   :", BATCH_SIZE)
print("VAL_BATCH_SIZE               :", VAL_BATCH_SIZE)
print("LEARNING_RATE                :", LEARNING_RATE)
print("WEIGHT_DECAY                 :", WEIGHT_DECAY)
print("SCORE_THRESHOLD              :", SCORE_THRESHOLD)
print("MASK_THRESHOLD               :", MASK_THRESHOLD)
print("VIDEO PROCESS_EVERY_N_FRAMES :", PROCESS_EVERY_N_FRAMES)
print("MAX_VIDEO_FRAMES             :", MAX_VIDEO_FRAMES)


## Cell 3 — Membuat Folder Output Otomatis

In [ ]:
# ============================================================
# Cell 3 - Create Output Directories
# ============================================================

for path in [
    DATA_DIR,
    IMAGE_DIR,
    ANNOTATION_DIR,
    VIDEO_DIR,
    CHECKPOINT_DIR,
    OUTPUT_DIR,
    LOG_DIR,
    PRED_DIR,
    METRIC_DIR
]:
    path.mkdir(parents=True, exist_ok=True)

print("Folder project siap.")

print("\nFolder penting:")
print("TRAIN_IMG_DIR :", TRAIN_IMG_DIR)
print("VAL_IMG_DIR   :", VAL_IMG_DIR)
print("TEST_IMG_DIR  :", TEST_IMG_DIR)
print("TRAIN_JSON    :", TRAIN_JSON)
print("VAL_JSON      :", VAL_JSON)
print("TEST_JSON     :", TEST_JSON)
print("VIDEO_DIR     :", VIDEO_DIR)


## Cell 4 — Dataset Loader COCO untuk Mask R-CNN

In [ ]:
# ============================================================
# Cell 4 - COCO Dataset Loader for Mask R-CNN
# ============================================================

class RandomHorizontalFlipDetection:
    """
    Augmentasi horizontal flip untuk object detection dan instance segmentation.
    Flip diterapkan ke image, bounding box, dan mask.
    """

    def __init__(self, prob=0.5):
        self.prob = prob

    def __call__(self, image, target):
        if random.random() < self.prob:
            image = torch.flip(image, dims=[2])
            _, h, w = image.shape

            boxes = target["boxes"]

            if boxes.numel() > 0:
                x_min = boxes[:, 0].clone()
                x_max = boxes[:, 2].clone()

                boxes[:, 0] = w - x_max
                boxes[:, 2] = w - x_min

                target["boxes"] = boxes

            if target["masks"].numel() > 0:
                target["masks"] = torch.flip(target["masks"], dims=[2])

        return image, target


class PotholeCocoDataset(Dataset):
    """
    Dataset loader untuk Mask R-CNN berbasis COCO JSON.

    Output:
        image  : Tensor [3, H, W], nilai 0 sampai 1
        target : dict:
            boxes    : Tensor [N, 4]
            labels   : Tensor [N]
            masks    : Tensor [N, H, W]
            image_id : Tensor [1]
            area     : Tensor [N]
            iscrowd  : Tensor [N]
    """

    def __init__(
        self,
        image_dir,
        annotation_file,
        input_size=416,
        transforms=None
    ):
        self.image_dir = Path(image_dir)
        self.annotation_file = Path(annotation_file)
        self.input_size = input_size
        self.transforms = transforms

        if not self.image_dir.exists():
            raise FileNotFoundError(f"Folder gambar tidak ditemukan: {self.image_dir}")

        if not self.annotation_file.exists():
            raise FileNotFoundError(f"File anotasi tidak ditemukan: {self.annotation_file}")

        self.coco = COCO(str(self.annotation_file))
        self.image_ids = sorted(self.coco.getImgIds())

        self.categories = self.coco.loadCats(self.coco.getCatIds())
        self.cat_id_to_name = {cat["id"]: cat["name"] for cat in self.categories}

        # Untuk satu kelas pothole, semua kategori objek dipetakan ke label 1.
        # Label 0 otomatis digunakan sebagai background oleh Mask R-CNN.
        self.category_mapping = {}
        for cat in self.categories:
            self.category_mapping[cat["id"]] = 1

        print(f"Dataset loaded      : {self.annotation_file}")
        print(f"Image directory     : {self.image_dir}")
        print(f"Total images        : {len(self.image_ids)}")
        print(f"COCO categories     : {self.cat_id_to_name}")

    def __len__(self):
        return len(self.image_ids)

    def _find_image_path(self, file_name):
        image_path = self.image_dir / file_name

        if image_path.exists():
            return image_path

        image_path_alt = self.image_dir / Path(file_name).name

        if image_path_alt.exists():
            return image_path_alt

        raise FileNotFoundError(f"Gambar tidak ditemukan: {self.image_dir / file_name}")

    def _load_image(self, image_info):
        file_name = image_info["file_name"]
        image_path = self._find_image_path(file_name)
        image = Image.open(image_path).convert("RGB")
        return image

    def _resize_image_boxes_masks(self, image, boxes, masks):
        original_w, original_h = image.size
        new_w, new_h = self.input_size, self.input_size

        image = image.resize((new_w, new_h), resample=Image.BILINEAR)

        scale_x = new_w / original_w
        scale_y = new_h / original_h

        if len(boxes) > 0:
            boxes[:, [0, 2]] = boxes[:, [0, 2]] * scale_x
            boxes[:, [1, 3]] = boxes[:, [1, 3]] * scale_y

            boxes[:, [0, 2]] = np.clip(boxes[:, [0, 2]], 0, new_w)
            boxes[:, [1, 3]] = np.clip(boxes[:, [1, 3]], 0, new_h)

        resized_masks = []

        for mask in masks:
            mask_img = Image.fromarray(mask.astype(np.uint8))
            mask_img = mask_img.resize((new_w, new_h), resample=Image.NEAREST)
            resized_masks.append(np.array(mask_img, dtype=np.uint8))

        if len(resized_masks) > 0:
            masks = np.stack(resized_masks, axis=0)
        else:
            masks = np.zeros((0, new_h, new_w), dtype=np.uint8)

        return image, boxes, masks

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_info = self.coco.loadImgs(image_id)[0]

        image = self._load_image(image_info)

        ann_ids = self.coco.getAnnIds(imgIds=image_id)
        anns = self.coco.loadAnns(ann_ids)

        boxes = []
        labels = []
        masks = []
        iscrowd = []

        for ann in anns:
            if "bbox" not in ann:
                continue

            x, y, w, h = ann["bbox"]

            if w <= 1 or h <= 1:
                continue

            xmin = x
            ymin = y
            xmax = x + w
            ymax = y + h

            boxes.append([xmin, ymin, xmax, ymax])

            category_id = ann["category_id"]
            label = self.category_mapping.get(category_id, 1)
            labels.append(label)

            mask = self.coco.annToMask(ann)
            masks.append(mask)

            iscrowd.append(ann.get("iscrowd", 0))

        boxes = np.array(boxes, dtype=np.float32)
        labels = np.array(labels, dtype=np.int64)
        iscrowd = np.array(iscrowd, dtype=np.int64)

        if len(boxes) == 0:
            boxes = np.zeros((0, 4), dtype=np.float32)
            labels = np.zeros((0,), dtype=np.int64)
            masks = np.zeros((0, image.size[1], image.size[0]), dtype=np.uint8)
            iscrowd = np.zeros((0,), dtype=np.int64)
        else:
            masks = np.stack(masks, axis=0).astype(np.uint8)

        image, boxes, masks = self._resize_image_boxes_masks(image, boxes, masks)

        if len(boxes) > 0:
            areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        else:
            areas = np.zeros((0,), dtype=np.float32)

        image = F.to_tensor(image)

        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32),
            "labels": torch.as_tensor(labels, dtype=torch.int64),
            "masks": torch.as_tensor(masks, dtype=torch.uint8),
            "image_id": torch.tensor([image_id], dtype=torch.int64),
            "area": torch.as_tensor(areas, dtype=torch.float32),
            "iscrowd": torch.as_tensor(iscrowd, dtype=torch.int64)
        }

        if self.transforms is not None:
            image, target = self.transforms(image, target)

        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))


## Cell 4.1 — Uji Dataset Loader dan Visualisasi Sample

In [ ]:
# ============================================================
# Cell 4.1 - Dataset Loader Test and Visualization
# ============================================================

train_transforms = RandomHorizontalFlipDetection(prob=AUG_HFLIP_PROB) if AUG_HFLIP_ENABLED else None
val_transforms = None

train_dataset = PotholeCocoDataset(
    image_dir=TRAIN_IMG_DIR,
    annotation_file=TRAIN_JSON,
    input_size=INPUT_SIZE,
    transforms=train_transforms
)

val_dataset = PotholeCocoDataset(
    image_dir=VAL_IMG_DIR,
    annotation_file=VAL_JSON,
    input_size=INPUT_SIZE,
    transforms=val_transforms
)

print("Jumlah data train:", len(train_dataset))
print("Jumlah data val  :", len(val_dataset))

sample_image, sample_target = train_dataset[0]

print("\nSample image shape :", sample_image.shape)
print("Boxes shape        :", sample_target["boxes"].shape)
print("Labels shape       :", sample_target["labels"].shape)
print("Masks shape        :", sample_target["masks"].shape)


def visualize_sample(dataset, index=0):
    image, target = dataset[index]

    image_np = image.permute(1, 2, 0).cpu().numpy()
    image_np = (image_np * 255).astype(np.uint8).copy()

    boxes = target["boxes"].cpu().numpy()
    masks = target["masks"].cpu().numpy()

    overlay = image_np.copy()

    for i in range(len(boxes)):
        x1, y1, x2, y2 = boxes[i].astype(int)

        cv2.rectangle(
            overlay,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        cv2.putText(
            overlay,
            "pothole",
            (x1, max(y1 - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2
        )

        mask = masks[i].astype(np.uint8)
        color_mask = np.zeros_like(image_np)
        color_mask[:, :, 1] = mask * 255

        overlay = cv2.addWeighted(overlay, 1.0, color_mask, 0.35, 0)

    plt.figure(figsize=(8, 8))
    plt.imshow(overlay)
    plt.axis("off")
    plt.title(f"Sample Index: {index} | Objects: {len(boxes)}")
    plt.show()


visualize_sample(train_dataset, index=0)


## Cell 5 — DataLoader Train dan Validation

In [ ]:
# ============================================================
# Cell 5 - DataLoader Train and Validation
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=PIN_MEMORY and torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=PIN_MEMORY and torch.cuda.is_available()
)

print("DataLoader siap.")
print(f"Jumlah batch train : {len(train_loader)}")
print(f"Jumlah batch val   : {len(val_loader)}")


images, targets = next(iter(train_loader))

print("\nUji satu batch:")
print("Tipe images :", type(images))
print("Tipe targets:", type(targets))
print("Jumlah image dalam batch:", len(images))
print("Image pertama shape:", images[0].shape)

print("\nTarget pertama:")
for key, value in targets[0].items():
    print(f"{key:10s}: shape={value.shape}, dtype={value.dtype}")


## Cell 6 — Model Mask R-CNN + MobileNetV3 + Feature Pyramid Network

In [ ]:
# ============================================================
# Cell 6 - Build Mask R-CNN + MobileNetV3-Large + FPN
# ============================================================

def build_mobilenetv3_fpn_backbone(
    pretrained_backbone=PRETRAINED_BACKBONE,
    trainable_backbone_layers=6
):
    try:
        from torchvision.models import MobileNet_V3_Large_Weights

        weights_backbone = (
            MobileNet_V3_Large_Weights.DEFAULT
            if pretrained_backbone
            else None
        )

        base_detector = fasterrcnn_mobilenet_v3_large_fpn(
            weights=None,
            weights_backbone=weights_backbone,
            trainable_backbone_layers=trainable_backbone_layers
        )

        print("Backbone MobileNetV3-FPN berhasil dibuat.")

    except Exception as e:
        print("Pretrained backbone gagal dimuat atau API TorchVision berbeda.")
        print("Detail error:", str(e))
        print("Mencoba membuat backbone tanpa pretrained weight.")

        try:
            base_detector = fasterrcnn_mobilenet_v3_large_fpn(
                weights=None,
                weights_backbone=None,
                trainable_backbone_layers=trainable_backbone_layers
            )
        except TypeError:
            base_detector = fasterrcnn_mobilenet_v3_large_fpn(
                pretrained=False,
                pretrained_backbone=False,
                trainable_backbone_layers=trainable_backbone_layers
            )

    backbone = base_detector.backbone

    print("Backbone out_channels:", backbone.out_channels)

    return backbone


def build_maskrcnn_mobilenetv3_fpn(
    num_classes=2,
    input_size=416,
    pretrained_backbone=PRETRAINED_BACKBONE,
    trainable_backbone_layers=TRAINABLE_BACKBONE_LAYERS,
    score_threshold=0.5
):
    backbone = build_mobilenetv3_fpn_backbone(
        pretrained_backbone=pretrained_backbone,
        trainable_backbone_layers=trainable_backbone_layers
    )

    model = MaskRCNN(
        backbone=backbone,
        num_classes=num_classes,
        min_size=input_size,
        max_size=input_size,
        rpn_pre_nms_top_n_train=RPN_PRE_NMS_TOP_N_TRAIN,
        rpn_post_nms_top_n_train=RPN_POST_NMS_TOP_N_TRAIN,
        rpn_pre_nms_top_n_test=RPN_PRE_NMS_TOP_N_TEST,
        rpn_post_nms_top_n_test=RPN_POST_NMS_TOP_N_TEST,
        box_detections_per_img=BOX_DETECTIONS_PER_IMG,
        box_score_thresh=score_threshold
    )

    return model


model = build_maskrcnn_mobilenetv3_fpn(
    num_classes=NUM_CLASSES,
    input_size=INPUT_SIZE,
    pretrained_backbone=PRETRAINED_BACKBONE,
    trainable_backbone_layers=TRAINABLE_BACKBONE_LAYERS,
    score_threshold=SCORE_THRESHOLD
)

model = model.to(device)

print("Model Mask R-CNN + MobileNetV3-FPN siap.")
print("Device:", device)


def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params = total_params - trainable_params
    return total_params, trainable_params, frozen_params


total_params, trainable_params, frozen_params = count_parameters(model)

print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"Frozen parameters    : {frozen_params:,}")


## Cell 6.1 — Uji Forward Pass Satu Batch

In [ ]:
# ============================================================
# Cell 6.1 - Test Forward Pass
# ============================================================

model.train()

images, targets = next(iter(train_loader))

images = [img.to(device) for img in images]
targets = [
    {k: v.to(device) for k, v in t.items()}
    for t in targets
]

loss_dict = model(images, targets)

print("Forward pass berhasil.")
print("\nLoss components:")

for key, value in loss_dict.items():
    print(f"{key:20s}: {value.item():.6f}")

total_loss = sum(loss for loss in loss_dict.values())
print(f"\nTotal loss: {total_loss.item():.6f}")


## Cell 7 — Optimizer, Scheduler, Checkpoint, dan Training Loop

In [ ]:
# ============================================================
# Cell 7 - Optimizer, Scheduler, Checkpoint, and Training Loop
# ============================================================

EPOCHS = NUM_EPOCHS
LR = LEARNING_RATE
WD = WEIGHT_DECAY

USE_AMP = USE_AMP_CONFIG and torch.cuda.is_available()
GRAD_CLIP_NORM = GRAD_CLIP_NORM if GRAD_CLIP_ENABLED else None

BEST_MODEL_PATH = CHECKPOINT_DIR / BEST_MODEL_NAME
LAST_MODEL_PATH = CHECKPOINT_DIR / LAST_MODEL_NAME
LOG_CSV_PATH = LOG_DIR / LOG_CSV_NAME


def autocast_context(enabled=True):
    if not enabled or not torch.cuda.is_available():
        return nullcontext()

    try:
        return torch.amp.autocast(device_type="cuda", enabled=enabled)
    except Exception:
        return torch.cuda.amp.autocast(enabled=enabled)


def create_grad_scaler(enabled=True):
    if not enabled or not torch.cuda.is_available():
        try:
            return torch.amp.GradScaler("cuda", enabled=False)
        except Exception:
            return torch.cuda.amp.GradScaler(enabled=False)

    try:
        return torch.amp.GradScaler("cuda", enabled=True)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=True)


params = [p for p in model.parameters() if p.requires_grad]

if OPTIMIZER_NAME.lower() == "adamw":
    optimizer = torch.optim.AdamW(
        params,
        lr=LR,
        weight_decay=WD
    )
elif OPTIMIZER_NAME.lower() == "sgd":
    optimizer = torch.optim.SGD(
        params,
        lr=LR,
        momentum=0.9,
        weight_decay=WD
    )
else:
    raise ValueError(f"Optimizer belum didukung: {OPTIMIZER_NAME}")

if SCHEDULER_NAME.lower() == "reduce_lr_on_plateau":
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode=SCHEDULER_MODE,
        factor=SCHEDULER_FACTOR,
        patience=SCHEDULER_PATIENCE
    )
else:
    raise ValueError(f"Scheduler belum didukung: {SCHEDULER_NAME}")

scaler = create_grad_scaler(enabled=USE_AMP)


def save_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    epoch,
    train_loss,
    val_loss,
    best_val_loss
):
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "best_val_loss": best_val_loss,
        "num_classes": NUM_CLASSES,
        "class_names": CLASS_NAMES,
        "input_size": INPUT_SIZE,
        "score_threshold": SCORE_THRESHOLD,
        "mask_threshold": MASK_THRESHOLD,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    torch.save(checkpoint, path)


def get_current_lr(optimizer):
    return optimizer.param_groups[0]["lr"]


def set_batchnorm_eval(module):
    if isinstance(module, torch.nn.modules.batchnorm._BatchNorm):
        module.eval()


def train_one_epoch(
    model,
    train_loader,
    optimizer,
    scaler,
    device,
    epoch,
    grad_clip_norm=5.0
):
    model.train()

    running = {
        "loss": 0.0,
        "loss_classifier": 0.0,
        "loss_box_reg": 0.0,
        "loss_mask": 0.0,
        "loss_objectness": 0.0,
        "loss_rpn_box_reg": 0.0
    }

    start_time = time.time()

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch} [Train]",
        leave=True
    )

    valid_batches = 0

    for batch_idx, (images, targets) in enumerate(progress_bar):
        images = [img.to(device) for img in images]
        targets = [
            {k: v.to(device) for k, v in t.items()}
            for t in targets
        ]

        optimizer.zero_grad(set_to_none=True)

        with autocast_context(enabled=USE_AMP):
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

        if not torch.isfinite(losses):
            print(f"Warning: loss tidak valid pada batch {batch_idx}: {losses.item()}")
            continue

        scaler.scale(losses).backward()
        scaler.unscale_(optimizer)

        if grad_clip_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)

        scaler.step(optimizer)
        scaler.update()

        valid_batches += 1

        running["loss"] += losses.item()

        for k in running.keys():
            if k != "loss":
                running[k] += loss_dict.get(k, torch.tensor(0.0, device=device)).item()

        avg_loss = running["loss"] / valid_batches

        progress_bar.set_postfix({
            "loss": f"{avg_loss:.4f}",
            "lr": f"{get_current_lr(optimizer):.2e}"
        })

    epoch_time = time.time() - start_time

    if valid_batches == 0:
        valid_batches = 1

    train_metrics = {
        "train_loss": running["loss"] / valid_batches,
        "train_loss_classifier": running["loss_classifier"] / valid_batches,
        "train_loss_box_reg": running["loss_box_reg"] / valid_batches,
        "train_loss_mask": running["loss_mask"] / valid_batches,
        "train_loss_objectness": running["loss_objectness"] / valid_batches,
        "train_loss_rpn_box_reg": running["loss_rpn_box_reg"] / valid_batches,
        "train_time_sec": epoch_time
    }

    return train_metrics


@torch.no_grad()
def validate_loss(
    model,
    val_loader,
    device,
    epoch
):
    model.train()
    model.apply(set_batchnorm_eval)

    running = {
        "loss": 0.0,
        "loss_classifier": 0.0,
        "loss_box_reg": 0.0,
        "loss_mask": 0.0,
        "loss_objectness": 0.0,
        "loss_rpn_box_reg": 0.0
    }

    start_time = time.time()

    progress_bar = tqdm(
        val_loader,
        desc=f"Epoch {epoch} [Val]",
        leave=True
    )

    valid_batches = 0

    for batch_idx, (images, targets) in enumerate(progress_bar):
        images = [img.to(device) for img in images]
        targets = [
            {k: v.to(device) for k, v in t.items()}
            for t in targets
        ]

        with autocast_context(enabled=USE_AMP):
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

        if not torch.isfinite(losses):
            print(f"Warning: val loss tidak valid pada batch {batch_idx}: {losses.item()}")
            continue

        valid_batches += 1

        running["loss"] += losses.item()

        for k in running.keys():
            if k != "loss":
                running[k] += loss_dict.get(k, torch.tensor(0.0, device=device)).item()

        avg_loss = running["loss"] / valid_batches
        progress_bar.set_postfix({"val_loss": f"{avg_loss:.4f}"})

    epoch_time = time.time() - start_time

    if valid_batches == 0:
        valid_batches = 1

    val_metrics = {
        "val_loss": running["loss"] / valid_batches,
        "val_loss_classifier": running["loss_classifier"] / valid_batches,
        "val_loss_box_reg": running["loss_box_reg"] / valid_batches,
        "val_loss_mask": running["loss_mask"] / valid_batches,
        "val_loss_objectness": running["loss_objectness"] / valid_batches,
        "val_loss_rpn_box_reg": running["loss_rpn_box_reg"] / valid_batches,
        "val_time_sec": epoch_time
    }

    return val_metrics


log_columns = [
    "epoch",
    "lr",
    "train_loss",
    "train_loss_classifier",
    "train_loss_box_reg",
    "train_loss_mask",
    "train_loss_objectness",
    "train_loss_rpn_box_reg",
    "train_time_sec",
    "val_loss",
    "val_loss_classifier",
    "val_loss_box_reg",
    "val_loss_mask",
    "val_loss_objectness",
    "val_loss_rpn_box_reg",
    "val_time_sec",
    "best_val_loss",
    "is_best"
]

with open(LOG_CSV_PATH, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=log_columns)
    writer.writeheader()

print("Training configuration:")
print(f"Epochs      : {EPOCHS}")
print(f"LR          : {LR}")
print(f"Weight decay: {WD}")
print(f"AMP         : {USE_AMP}")
print(f"Best path   : {BEST_MODEL_PATH}")
print(f"Last path   : {LAST_MODEL_PATH}")
print(f"Log path    : {LOG_CSV_PATH}")


## Cell 7.1 — Jalankan Training

In [ ]:
# ============================================================
# Cell 7.1 - Run Main Training Loop
# ============================================================

best_val_loss = float("inf")
history = []

print("Mulai training...")
print("=" * 80)

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    train_metrics = train_one_epoch(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        scaler=scaler,
        device=device,
        epoch=epoch,
        grad_clip_norm=GRAD_CLIP_NORM
    )

    val_metrics = validate_loss(
        model=model,
        val_loader=val_loader,
        device=device,
        epoch=epoch
    )

    val_loss = val_metrics["val_loss"]

    scheduler.step(val_loss)

    is_best = val_loss < best_val_loss

    if is_best:
        best_val_loss = val_loss

        save_checkpoint(
            path=BEST_MODEL_PATH,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            epoch=epoch,
            train_loss=train_metrics["train_loss"],
            val_loss=val_loss,
            best_val_loss=best_val_loss
        )

    save_checkpoint(
        path=LAST_MODEL_PATH,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        epoch=epoch,
        train_loss=train_metrics["train_loss"],
        val_loss=val_loss,
        best_val_loss=best_val_loss
    )

    lr_now = get_current_lr(optimizer)

    log_row = {
        "epoch": epoch,
        "lr": lr_now,
        **train_metrics,
        **val_metrics,
        "best_val_loss": best_val_loss,
        "is_best": is_best
    }

    history.append(log_row)

    with open(LOG_CSV_PATH, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=log_columns)
        writer.writerow(log_row)

    epoch_time = time.time() - epoch_start

    print("-" * 80)
    print(f"Epoch [{epoch}/{EPOCHS}] selesai")
    print(f"Train Loss    : {train_metrics['train_loss']:.6f}")
    print(f"Val Loss      : {val_metrics['val_loss']:.6f}")
    print(f"Best Val Loss : {best_val_loss:.6f}")
    print(f"Learning Rate : {lr_now:.8f}")
    print(f"Is Best       : {is_best}")
    print(f"Epoch Time    : {epoch_time:.2f} sec")
    print("-" * 80)

print("=" * 80)
print("Training selesai.")
print(f"Best model disimpan di: {BEST_MODEL_PATH}")
print(f"Last model disimpan di: {LAST_MODEL_PATH}")
print(f"Training log disimpan di: {LOG_CSV_PATH}")


## Cell 7.2 — Plot Loss

In [ ]:
# ============================================================
# Cell 7.2 - Plot Training and Validation Loss
# ============================================================

history_df = pd.DataFrame(history)
history_df.to_csv(LOG_CSV_PATH, index=False)

display(history_df.tail())

plt.figure(figsize=(10, 6))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train Loss")
plt.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Mask R-CNN MobileNetV3-FPN Training Loss")
plt.legend()
plt.grid(True)

loss_plot_path = LOG_DIR / LOSS_PLOT_NAME
plt.savefig(loss_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Plot loss disimpan di: {loss_plot_path}")


## Cell 8 — Load Best Model untuk Inferensi

In [ ]:
# ============================================================
# Cell 8 - Load Best Model for Inference
# ============================================================

BEST_MODEL_PATH = CHECKPOINT_DIR / BEST_MODEL_NAME

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f"Best model tidak ditemukan: {BEST_MODEL_PATH}")

checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)

print("Checkpoint berhasil dimuat.")
print("Epoch terbaik       :", checkpoint.get("epoch"))
print("Best validation loss:", checkpoint.get("best_val_loss"))
print("Input size          :", checkpoint.get("input_size"))
print("Class names         :", checkpoint.get("class_names"))

inference_model = build_maskrcnn_mobilenetv3_fpn(
    num_classes=checkpoint.get("num_classes", NUM_CLASSES),
    input_size=checkpoint.get("input_size", INPUT_SIZE),
    pretrained_backbone=False,
    trainable_backbone_layers=TRAINABLE_BACKBONE_LAYERS,
    score_threshold=SCORE_THRESHOLD
)

inference_model.load_state_dict(checkpoint["model_state_dict"])
inference_model = inference_model.to(device)
inference_model.eval()

print("Model terbaik siap untuk inferensi.")


## Cell 9 — Fungsi Inferensi Satu Gambar

In [ ]:
# ============================================================
# Cell 9 - Single Image Inference Function
# ============================================================

def preprocess_image_for_inference(image_path, input_size=416):
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Gambar tidak ditemukan: {image_path}")

    image = Image.open(image_path).convert("RGB")
    image = image.resize((input_size, input_size), resample=Image.BILINEAR)

    image_np = np.array(image).copy()
    image_tensor = F.to_tensor(image)

    return image_tensor, image_np


@torch.no_grad()
def predict_single_image(
    model,
    image_path,
    device,
    input_size=416,
    score_threshold=0.5,
    mask_threshold=0.5
):
    model.eval()

    image_tensor, image_np = preprocess_image_for_inference(
        image_path=image_path,
        input_size=input_size
    )

    image_tensor = image_tensor.to(device)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start_time = time.time()

    prediction = model([image_tensor])[0]

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    inference_time = time.time() - start_time

    boxes = prediction["boxes"].detach().cpu()
    scores = prediction["scores"].detach().cpu()
    labels = prediction["labels"].detach().cpu()
    masks = prediction["masks"].detach().cpu()

    keep = scores >= score_threshold

    boxes = boxes[keep]
    scores = scores[keep]
    labels = labels[keep]
    masks = masks[keep]

    if len(masks) > 0:
        masks = masks[:, 0, :, :]
        masks = masks >= mask_threshold
    else:
        masks = torch.zeros((0, input_size, input_size), dtype=torch.bool)

    result = {
        "image_np": image_np,
        "boxes": boxes,
        "scores": scores,
        "labels": labels,
        "masks": masks,
        "inference_time": inference_time
    }

    return result


## Cell 10 — Visualisasi Prediksi Gambar

In [ ]:
# ============================================================
# Cell 10 - Visualize Single Image Prediction
# ============================================================

def visualize_prediction(
    result,
    save_path=None,
    class_names=None
):
    image_np = result["image_np"].copy()
    boxes = result["boxes"].numpy()
    scores = result["scores"].numpy()
    labels = result["labels"].numpy()
    masks = result["masks"].numpy()

    overlay = image_np.copy()

    for i in range(len(boxes)):
        x1, y1, x2, y2 = boxes[i].astype(int)

        label_id = int(labels[i])
        score = float(scores[i])

        if class_names is not None and label_id < len(class_names):
            label_name = class_names[label_id]
        else:
            label_name = f"class_{label_id}"

        cv2.rectangle(
            overlay,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        text = f"{label_name}: {score:.2f}"

        cv2.putText(
            overlay,
            text,
            (x1, max(y1 - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 255, 0),
            2
        )

        mask = masks[i].astype(np.uint8)
        color_mask = np.zeros_like(image_np)
        color_mask[:, :, 1] = mask * 255

        overlay = cv2.addWeighted(
            overlay,
            1.0,
            color_mask,
            0.35,
            0
        )

    plt.figure(figsize=(8, 8))
    plt.imshow(overlay)
    plt.axis("off")
    plt.title(
        f"Detected objects: {len(boxes)} | "
        f"Inference time: {result['inference_time']:.4f} sec"
    )
    plt.show()

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)

        overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
        cv2.imwrite(str(save_path), overlay_bgr)

        print(f"Hasil prediksi disimpan di: {save_path}")


test_images = sorted(
    list(TEST_IMG_DIR.glob("*.jpg")) +
    list(TEST_IMG_DIR.glob("*.jpeg")) +
    list(TEST_IMG_DIR.glob("*.png"))
)

if len(test_images) == 0:
    raise FileNotFoundError(f"Tidak ada gambar test di folder: {TEST_IMG_DIR}")

sample_test_image = test_images[0]

print("Gambar yang diuji:", sample_test_image)

result = predict_single_image(
    model=inference_model,
    image_path=sample_test_image,
    device=device,
    input_size=INPUT_SIZE,
    score_threshold=SCORE_THRESHOLD,
    mask_threshold=MASK_THRESHOLD
)

print("Jumlah objek terdeteksi:", len(result["boxes"]))
print("Inference time:", result["inference_time"], "detik")

save_path = PRED_DIR / f"{sample_test_image.stem}{PREDICTION_IMAGE_SUFFIX}"

visualize_prediction(
    result=result,
    save_path=save_path,
    class_names=CLASS_NAMES
)


## Cell 11 — Batch Inferensi Folder Test

In [ ]:
# ============================================================
# Cell 11 - Batch Inference on Test Images
# ============================================================

def run_batch_inference(
    model,
    image_paths,
    device,
    output_dir,
    input_size=416,
    score_threshold=0.5,
    mask_threshold=0.5,
    class_names=None,
    max_images=MAX_TEST_IMAGES
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    records = []

    selected_images = image_paths[:max_images]

    for image_path in tqdm(selected_images, desc="Batch inference"):
        result = predict_single_image(
            model=model,
            image_path=image_path,
            device=device,
            input_size=input_size,
            score_threshold=score_threshold,
            mask_threshold=mask_threshold
        )

        save_path = output_dir / f"{Path(image_path).stem}{PREDICTION_IMAGE_SUFFIX}"

        visualize_prediction(
            result=result,
            save_path=save_path,
            class_names=class_names
        )

        boxes = result["boxes"]
        scores = result["scores"]
        labels = result["labels"]

        for i in range(len(boxes)):
            x1, y1, x2, y2 = boxes[i].tolist()
            label_id = int(labels[i].item())

            records.append({
                "image_name": Path(image_path).name,
                "label": label_id,
                "label_name": class_names[label_id] if class_names else label_id,
                "score": float(scores[i].item()),
                "x_min": x1,
                "y_min": y1,
                "x_max": x2,
                "y_max": y2,
                "inference_time_sec": result["inference_time"]
            })

        if len(boxes) == 0:
            records.append({
                "image_name": Path(image_path).name,
                "label": None,
                "label_name": None,
                "score": None,
                "x_min": None,
                "y_min": None,
                "x_max": None,
                "y_max": None,
                "inference_time_sec": result["inference_time"]
            })

    result_df = pd.DataFrame(records)

    csv_path = output_dir / "batch_inference_results.csv"
    result_df.to_csv(csv_path, index=False)

    print(f"CSV hasil inferensi disimpan di: {csv_path}")

    return result_df


batch_output_dir = PRED_DIR / "test_batch_predictions"

batch_results_df = run_batch_inference(
    model=inference_model,
    image_paths=test_images,
    device=device,
    output_dir=batch_output_dir,
    input_size=INPUT_SIZE,
    score_threshold=SCORE_THRESHOLD,
    mask_threshold=MASK_THRESHOLD,
    class_names=CLASS_NAMES,
    max_images=MAX_TEST_IMAGES
)

display(batch_results_df.head())


## Cell 12 — Inferensi Video

In [ ]:
# ============================================================
# Cell 12 - Video Inference for Mask R-CNN Pothole Detection
# ============================================================

def preprocess_frame_for_inference(frame_bgr, input_size=416):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    resized_rgb = cv2.resize(
        frame_rgb,
        (input_size, input_size),
        interpolation=cv2.INTER_LINEAR
    )

    image_tensor = torch.from_numpy(resized_rgb).permute(2, 0, 1).float() / 255.0

    return image_tensor, resized_rgb


@torch.no_grad()
def predict_video_frame(
    model,
    frame_bgr,
    device,
    input_size=416,
    score_threshold=0.5,
    mask_threshold=0.5
):
    model.eval()

    original_h, original_w = frame_bgr.shape[:2]

    image_tensor, resized_rgb = preprocess_frame_for_inference(
        frame_bgr=frame_bgr,
        input_size=input_size
    )

    image_tensor = image_tensor.to(device)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start_time = time.time()

    prediction = model([image_tensor])[0]

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    inference_time = time.time() - start_time

    boxes = prediction["boxes"].detach().cpu()
    scores = prediction["scores"].detach().cpu()
    labels = prediction["labels"].detach().cpu()
    masks = prediction["masks"].detach().cpu()

    keep = scores >= score_threshold

    boxes = boxes[keep]
    scores = scores[keep]
    labels = labels[keep]
    masks = masks[keep]

    if len(masks) > 0:
        masks = masks[:, 0, :, :]
        masks = masks >= mask_threshold
    else:
        masks = torch.zeros((0, input_size, input_size), dtype=torch.bool)

    result = {
        "boxes": boxes,
        "scores": scores,
        "labels": labels,
        "masks": masks,
        "inference_time": inference_time,
        "input_size": input_size,
        "original_size": (original_w, original_h)
    }

    return result


def draw_predictions_on_frame(
    frame_bgr,
    result,
    class_names=None,
    mask_alpha=MASK_ALPHA
):
    output_frame = frame_bgr.copy()

    original_h, original_w = output_frame.shape[:2]
    input_size = result["input_size"]

    scale_x = original_w / input_size
    scale_y = original_h / input_size

    boxes = result["boxes"].numpy()
    scores = result["scores"].numpy()
    labels = result["labels"].numpy()
    masks = result["masks"].numpy()

    overlay = output_frame.copy()

    for i in range(len(boxes)):
        x1, y1, x2, y2 = boxes[i]

        x1 = int(x1 * scale_x)
        y1 = int(y1 * scale_y)
        x2 = int(x2 * scale_x)
        y2 = int(y2 * scale_y)

        label_id = int(labels[i])
        score = float(scores[i])

        if class_names is not None and label_id < len(class_names):
            label_name = class_names[label_id]
        else:
            label_name = f"class_{label_id}"

        mask = masks[i].astype(np.uint8) * 255

        mask_resized = cv2.resize(
            mask,
            (original_w, original_h),
            interpolation=cv2.INTER_NEAREST
        )

        color_mask = np.zeros_like(output_frame)
        color_mask[:, :, 1] = mask_resized

        overlay = cv2.addWeighted(
            overlay,
            1.0,
            color_mask,
            mask_alpha,
            0
        )

        cv2.rectangle(
            overlay,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        text = f"{label_name}: {score:.2f}"

        cv2.putText(
            overlay,
            text,
            (x1, max(y1 - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2
        )

    inference_time = result["inference_time"]
    fps = 1.0 / inference_time if inference_time > 0 else 0.0

    info_text = f"Inference: {inference_time:.3f}s | FPS: {fps:.2f} | Objects: {len(boxes)}"

    cv2.putText(
        overlay,
        info_text,
        (20, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 255),
        2
    )

    return overlay


def run_video_inference(
    model,
    video_path,
    output_path,
    device,
    input_size=416,
    score_threshold=0.5,
    mask_threshold=0.5,
    class_names=None,
    process_every_n_frames=1,
    max_frames=None
):
    video_path = Path(video_path)
    output_path = Path(output_path)

    if not video_path.exists():
        raise FileNotFoundError(f"Video tidak ditemukan: {video_path}")

    output_path.parent.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(f"Gagal membuka video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    original_fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    if original_fps <= 0:
        original_fps = 25.0

    if max_frames is not None:
        total_to_process = min(total_frames, max_frames)
    else:
        total_to_process = total_frames

    fourcc = cv2.VideoWriter_fourcc(*VIDEO_CODEC)

    writer = cv2.VideoWriter(
        str(output_path),
        fourcc,
        original_fps,
        (width, height)
    )

    print("Video input             :", video_path)
    print("Video output            :", output_path)
    print("Resolution              :", width, "x", height)
    print("FPS original            :", original_fps)
    print("Total frame             :", total_frames)
    print("Frame target proses     :", total_to_process)
    print("Process every N frames  :", process_every_n_frames)

    frame_index = 0
    processed_count = 0
    inference_times = []
    last_result = None

    progress_bar = tqdm(total=total_to_process, desc="Video inference")

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        if max_frames is not None and frame_index >= max_frames:
            break

        if frame_index % process_every_n_frames == 0 or last_result is None:
            result = predict_video_frame(
                model=model,
                frame_bgr=frame,
                device=device,
                input_size=input_size,
                score_threshold=score_threshold,
                mask_threshold=mask_threshold
            )

            last_result = result
            inference_times.append(result["inference_time"])
            processed_count += 1
        else:
            result = last_result

        output_frame = draw_predictions_on_frame(
            frame_bgr=frame,
            result=result,
            class_names=class_names
        )

        writer.write(output_frame)

        frame_index += 1
        progress_bar.update(1)

    progress_bar.close()

    cap.release()
    writer.release()

    avg_inference_time = np.mean(inference_times) if len(inference_times) > 0 else 0.0
    avg_fps_model = 1.0 / avg_inference_time if avg_inference_time > 0 else 0.0

    print("\nInferensi video selesai.")
    print(f"Output video disimpan di          : {output_path}")
    print(f"Total frame dibaca                : {frame_index}")
    print(f"Frame diprediksi model            : {processed_count}")
    print(f"Average inference time            : {avg_inference_time:.4f} sec/frame")
    print(f"Approx model FPS                  : {avg_fps_model:.2f}")

    summary = {
        "video_path": str(video_path),
        "output_path": str(output_path),
        "total_frames_read": frame_index,
        "processed_frames_by_model": processed_count,
        "process_every_n_frames": process_every_n_frames,
        "avg_inference_time_sec": avg_inference_time,
        "approx_model_fps": avg_fps_model,
        "input_size": input_size,
        "score_threshold": score_threshold,
        "mask_threshold": mask_threshold,
        "width": width,
        "height": height,
        "original_fps": original_fps
    }

    return summary


## Cell 12.1 — Jalankan Inferensi Video

In [ ]:
# ============================================================
# Cell 12.1 - Run Video File Inference
# ============================================================

input_video_path = VIDEO_DIR / INPUT_VIDEO_NAME
output_video_path = PRED_DIR / OUTPUT_VIDEO_NAME

video_summary = run_video_inference(
    model=inference_model,
    video_path=input_video_path,
    output_path=output_video_path,
    device=device,
    input_size=INPUT_SIZE,
    score_threshold=SCORE_THRESHOLD,
    mask_threshold=MASK_THRESHOLD,
    class_names=CLASS_NAMES,

    # Untuk laptop GPU kuat: 1.
    # Untuk laptop CPU atau Jetson Nano: 3 sampai 5.
    process_every_n_frames=PROCESS_EVERY_N_FRAMES,

    # Uji awal cukup 100 frame.
    # Jika sudah aman, ubah menjadi None.
    max_frames=MAX_VIDEO_FRAMES
)

video_summary_df = pd.DataFrame([video_summary])
video_summary_path = PRED_DIR / "video_inference_summary.csv"
video_summary_df.to_csv(video_summary_path, index=False)

display(video_summary_df)

print(f"Ringkasan inferensi video disimpan di: {video_summary_path}")


## Cell 13 — Evaluasi Kuantitatif Test Set

Menghitung TP, FP, FN, precision, recall, F1-score, Box IoU, Mask IoU, latency, dan FPS.

In [ ]:
from torchvision.ops import box_iou
import json

if "test_dataset" not in globals():
    test_dataset = PotholeCocoDataset(TEST_IMG_DIR, TEST_JSON, input_size=INPUT_SIZE, transforms=None)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=PIN_MEMORY and torch.cuda.is_available()
)

def compute_mask_iou(pred_mask, gt_mask):
    pred_mask = pred_mask.cpu().numpy() if isinstance(pred_mask, torch.Tensor) else pred_mask
    gt_mask = gt_mask.cpu().numpy() if isinstance(gt_mask, torch.Tensor) else gt_mask
    pred_mask = pred_mask.astype(bool)
    gt_mask = gt_mask.astype(bool)
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()
    return 0.0 if union == 0 else float(intersection / union)

@torch.no_grad()
def evaluate_single_image(model, image, target, device, score_threshold=0.5, mask_threshold=0.5, iou_threshold=0.5):
    model.eval()
    if torch.cuda.is_available(): torch.cuda.synchronize()
    start_time = time.time()
    prediction = model([image.to(device)])[0]
    if torch.cuda.is_available(): torch.cuda.synchronize()
    inference_time = time.time() - start_time

    pred_boxes = prediction["boxes"].detach().cpu()
    pred_scores = prediction["scores"].detach().cpu()
    pred_masks = prediction["masks"].detach().cpu()
    keep = pred_scores >= score_threshold
    pred_boxes, pred_scores, pred_masks = pred_boxes[keep], pred_scores[keep], pred_masks[keep]
    pred_masks = pred_masks[:, 0, :, :] >= mask_threshold if len(pred_masks) > 0 else torch.zeros((0, INPUT_SIZE, INPUT_SIZE), dtype=torch.bool)

    gt_boxes = target["boxes"].detach().cpu()
    gt_masks = target["masks"].detach().cpu().bool()
    if len(pred_boxes) == 0:
        return {"tp": 0, "fp": 0, "fn": len(gt_boxes), "box_ious": [], "mask_ious": [], "num_predictions": 0, "num_ground_truth": len(gt_boxes), "inference_time": inference_time}
    if len(gt_boxes) == 0:
        return {"tp": 0, "fp": len(pred_boxes), "fn": 0, "box_ious": [], "mask_ious": [], "num_predictions": len(pred_boxes), "num_ground_truth": 0, "inference_time": inference_time}

    iou_matrix = box_iou(pred_boxes, gt_boxes)
    matched_gt = set()
    tp, fp = 0, 0
    matched_box_ious, matched_mask_ious = [], []
    for pred_idx in torch.argsort(pred_scores, descending=True):
        pred_idx = int(pred_idx.item())
        best_gt_idx = int(torch.argmax(iou_matrix[pred_idx]).item())
        best_box_iou = float(iou_matrix[pred_idx, best_gt_idx].item())
        best_mask_iou = compute_mask_iou(pred_masks[pred_idx], gt_masks[best_gt_idx])
        if best_box_iou >= iou_threshold and best_mask_iou >= iou_threshold and best_gt_idx not in matched_gt:
            tp += 1
            matched_gt.add(best_gt_idx)
            matched_box_ious.append(best_box_iou)
            matched_mask_ious.append(best_mask_iou)
        else:
            fp += 1
    fn = len(gt_boxes) - len(matched_gt)
    return {"tp": tp, "fp": fp, "fn": fn, "box_ious": matched_box_ious, "mask_ious": matched_mask_ious, "num_predictions": len(pred_boxes), "num_ground_truth": len(gt_boxes), "inference_time": inference_time}


## Cell 13.1 — Jalankan Evaluasi Kuantitatif

In [ ]:
IOU_THRESHOLD = 0.5
records = []
total_tp = total_fp = total_fn = 0
all_box_ious, all_mask_ious, all_times = [], [], []

for idx, (images, targets) in enumerate(tqdm(test_loader, desc="Evaluating test set")):
    result = evaluate_single_image(inference_model, images[0], targets[0], device, SCORE_THRESHOLD, MASK_THRESHOLD, IOU_THRESHOLD)
    total_tp += result["tp"]
    total_fp += result["fp"]
    total_fn += result["fn"]
    all_box_ious.extend(result["box_ious"])
    all_mask_ious.extend(result["mask_ious"])
    all_times.append(result["inference_time"])
    records.append({"index": idx, "image_id": int(targets[0]["image_id"].item()), **{k: result[k] for k in ["tp", "fp", "fn", "num_predictions", "num_ground_truth", "inference_time"]}, "mean_box_iou_image": float(np.mean(result["box_ious"])) if result["box_ious"] else 0.0, "mean_mask_iou_image": float(np.mean(result["mask_ious"])) if result["mask_ious"] else 0.0})

precision = total_tp / (total_tp + total_fp) if total_tp + total_fp > 0 else 0.0
recall = total_tp / (total_tp + total_fn) if total_tp + total_fn > 0 else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
summary = {"iou_threshold": IOU_THRESHOLD, "score_threshold": SCORE_THRESHOLD, "mask_threshold": MASK_THRESHOLD, "total_images": len(test_dataset), "total_tp": total_tp, "total_fp": total_fp, "total_fn": total_fn, "precision": precision, "recall": recall, "f1_score": f1, "mean_box_iou": float(np.mean(all_box_ious)) if all_box_ious else 0.0, "mean_mask_iou": float(np.mean(all_mask_ious)) if all_mask_ious else 0.0, "avg_inference_time_sec": float(np.mean(all_times)) if all_times else 0.0}
summary["approx_fps"] = 1.0 / summary["avg_inference_time_sec"] if summary["avg_inference_time_sec"] > 0 else 0.0

METRIC_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame(records).to_csv(METRIC_DIR / "test_evaluation_detail.csv", index=False)
pd.DataFrame([summary]).to_csv(METRIC_DIR / "test_evaluation_summary.csv", index=False)
with open(METRIC_DIR / "test_evaluation_summary.json", "w", encoding="utf-8") as f: json.dump(summary, f, indent=4)
display(pd.DataFrame([summary]))


## Cell 14 — COCO mAP Evaluation

Menghitung bbox mAP dan segmentation mAP menggunakan pycocotools COCOeval.

In [ ]:
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as mask_utils

if "test_dataset" not in globals():
    test_dataset = PotholeCocoDataset(TEST_IMG_DIR, TEST_JSON, input_size=INPUT_SIZE, transforms=None)

DEFAULT_CATEGORY_ID = test_dataset.coco.getCatIds()[0]

def encode_binary_mask_to_rle(binary_mask):
    binary_mask = np.asfortranarray(binary_mask.astype(np.uint8))
    rle = mask_utils.encode(binary_mask)
    rle["counts"] = rle["counts"].decode("utf-8")
    return rle

@torch.no_grad()
def generate_coco_predictions(model, dataset, device, score_threshold=0.05, mask_threshold=0.5):
    model.eval(); bbox_results, segm_results, times = [], [], []
    for idx in tqdm(range(len(dataset)), desc="Generating COCO predictions"):
        image, target = dataset[idx]
        image_id = int(target["image_id"].item())
        info = dataset.coco.loadImgs(image_id)[0]
        original_w, original_h = int(info["width"]), int(info["height"])
        scale_x, scale_y = original_w / INPUT_SIZE, original_h / INPUT_SIZE
        if torch.cuda.is_available(): torch.cuda.synchronize()
        start = time.time(); pred = model([image.to(device)])[0]
        if torch.cuda.is_available(): torch.cuda.synchronize()
        times.append(time.time() - start)
        boxes, scores, masks = pred["boxes"].detach().cpu(), pred["scores"].detach().cpu(), pred["masks"].detach().cpu()
        keep = scores >= score_threshold
        boxes, scores, masks = boxes[keep], scores[keep], masks[keep]
        masks = masks[:, 0, :, :] >= mask_threshold if len(masks) else torch.zeros((0, INPUT_SIZE, INPUT_SIZE), dtype=torch.bool)
        for i in range(len(boxes)):
            x1, y1, x2, y2 = boxes[i].numpy()
            x1, x2 = float(x1 * scale_x), float(x2 * scale_x)
            y1, y2 = float(y1 * scale_y), float(y2 * scale_y)
            w, h = max(0.0, x2 - x1), max(0.0, y2 - y1)
            if w <= 1 or h <= 1: continue
            score = float(scores[i].item())
            bbox_results.append({"image_id": image_id, "category_id": DEFAULT_CATEGORY_ID, "bbox": [x1, y1, w, h], "score": score})
            mask_original = cv2.resize(masks[i].numpy().astype(np.uint8), (original_w, original_h), interpolation=cv2.INTER_NEAREST)
            segm_results.append({"image_id": image_id, "category_id": DEFAULT_CATEGORY_ID, "segmentation": encode_binary_mask_to_rle(mask_original), "score": score})
    speed = {"total_images": len(dataset), "avg_inference_time_sec": float(np.mean(times)) if times else 0.0}
    speed["approx_fps"] = 1.0 / speed["avg_inference_time_sec"] if speed["avg_inference_time_sec"] > 0 else 0.0
    return bbox_results, segm_results, speed

def run_coco_eval(coco_gt, json_path, iou_type):
    results = json.load(open(json_path, "r", encoding="utf-8"))
    if not results: return None, {f"{iou_type}_mAP": 0.0, f"{iou_type}_AP50": 0.0, f"{iou_type}_AP75": 0.0}
    coco_dt = coco_gt.loadRes(str(json_path))
    evaluator = COCOeval(coco_gt, coco_dt, iouType=iou_type)
    evaluator.evaluate(); evaluator.accumulate(); evaluator.summarize()
    s = evaluator.stats
    return evaluator, {f"{iou_type}_mAP": float(s[0]), f"{iou_type}_AP50": float(s[1]), f"{iou_type}_AP75": float(s[2]), f"{iou_type}_AP_small": float(s[3]), f"{iou_type}_AP_medium": float(s[4]), f"{iou_type}_AP_large": float(s[5])}

COCO_SCORE_THRESHOLD = 0.05
bbox_results, segm_results, speed_summary = generate_coco_predictions(inference_model, test_dataset, device, COCO_SCORE_THRESHOLD, MASK_THRESHOLD)
bbox_json, segm_json = METRIC_DIR / "coco_bbox_predictions.json", METRIC_DIR / "coco_segm_predictions.json"
json.dump(bbox_results, open(bbox_json, "w", encoding="utf-8")); json.dump(segm_results, open(segm_json, "w", encoding="utf-8"))
_, bbox_metrics = run_coco_eval(test_dataset.coco, bbox_json, "bbox")
_, segm_metrics = run_coco_eval(test_dataset.coco, segm_json, "segm")
coco_summary = {"score_threshold_for_coco_eval": COCO_SCORE_THRESHOLD, "mask_threshold": MASK_THRESHOLD, **bbox_metrics, **segm_metrics, **speed_summary}
with open(METRIC_DIR / "coco_map_summary.json", "w", encoding="utf-8") as f: json.dump(coco_summary, f, indent=4)
pd.DataFrame([coco_summary]).to_csv(METRIC_DIR / "coco_map_summary.csv", index=False)
display(pd.DataFrame([coco_summary]))


## Cell 15 — Analisis Ukuran Model, Latency, FPS, dan Estimasi Kelayakan Jetson Orin Nano

In [ ]:
import tempfile

def count_model_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total_parameters": int(total), "trainable_parameters": int(trainable), "frozen_parameters": int(total - trainable)}

def file_size_mb(path):
    path = Path(path); return path.stat().st_size / (1024 ** 2) if path.exists() else 0.0

def estimate_state_dict_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as tmp: tmp_path = Path(tmp.name)
    try:
        torch.save(model.state_dict(), tmp_path); return file_size_mb(tmp_path)
    finally:
        tmp_path.unlink(missing_ok=True)

parameter_info = count_model_parameters(inference_model)
checkpoint_size_mb = file_size_mb(BEST_MODEL_PATH)
state_dict_size_mb = estimate_state_dict_size_mb(inference_model)

@torch.no_grad()
def profile_model_latency(model, dataset, max_images=50, warmup_runs=5):
    model.eval(); total = min(len(dataset), max_images)
    for i in range(min(warmup_runs, total)):
        image, _ = dataset[i]; _ = model([image.to(device)])
    if torch.cuda.is_available(): torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    times, counts = [], []
    for i in tqdm(range(total), desc="Profiling model latency"):
        image, _ = dataset[i]
        if torch.cuda.is_available(): torch.cuda.synchronize()
        start = time.time(); pred = model([image.to(device)])[0]
        if torch.cuda.is_available(): torch.cuda.synchronize()
        times.append(time.time() - start); counts.append(int((pred["scores"].detach().cpu() >= SCORE_THRESHOLD).sum().item()))
    arr = np.array(times, dtype=np.float32)
    peak = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0
    return {"profiled_images": int(total), "avg_latency_sec": float(arr.mean()) if len(arr) else 0.0, "median_latency_sec": float(np.median(arr)) if len(arr) else 0.0, "avg_fps": float(1.0 / arr.mean()) if len(arr) and arr.mean() > 0 else 0.0, "avg_predictions_per_image": float(np.mean(counts)) if counts else 0.0, "peak_gpu_memory_mb": float(peak), "device": str(device), "input_size": int(INPUT_SIZE)}, times

model_latency_summary, latency_records = profile_model_latency(inference_model, test_dataset, max_images=min(50, len(test_dataset)))
latency_ms = np.array(latency_records) * 1000.0
plt.figure(figsize=(10, 6)); plt.hist(latency_ms, bins=20); plt.xlabel("Latency per image (ms)"); plt.ylabel("Frequency"); plt.title("Model-Only Inference Latency Distribution"); plt.grid(True)
latency_plot_path = METRIC_DIR / "latency_distribution_model_only.png"
plt.savefig(latency_plot_path, dpi=300, bbox_inches="tight"); plt.show()

def estimate_jetson_orin_feasibility(state_dict_size_mb, total_parameters, input_size, avg_fps, peak_gpu_memory_mb=0.0):
    risk = 0
    if state_dict_size_mb > 250: risk += 2
    elif state_dict_size_mb > 100: risk += 1
    if total_parameters > 30_000_000: risk += 2
    elif total_parameters > 15_000_000: risk += 1
    if input_size > 416: risk += 2
    elif input_size > 320: risk += 1
    if avg_fps < 5: risk += 2
    elif avg_fps < 15: risk += 1
    if peak_gpu_memory_mb > 3000: risk += 2
    elif peak_gpu_memory_mb > 1500: risk += 1
    feasibility = "LAYAK untuk uji awal Jetson Orin Nano" if risk <= 2 else ("CUKUP LAYAK, tetapi perlu ONNX/TensorRT/FP16 dan profiling device" if risk <= 5 else "BERISIKO tanpa optimasi tambahan")
    return {"feasibility": feasibility, "risk_score": risk, "recommendations": ["Gunakan batch_size=1 saat inferensi", "Bandingkan input_size 416 dan 320", "Aktifkan sudo nvpmodel -m 0 dan sudo jetson_clocks", "Pantau suhu/memori dengan jtop"]}

jetson_estimation = estimate_jetson_orin_feasibility(state_dict_size_mb, parameter_info["total_parameters"], INPUT_SIZE, model_latency_summary["avg_fps"], model_latency_summary["peak_gpu_memory_mb"])
profiling_summary = {"model": {"architecture": MODEL_ARCHITECTURE, "backbone": MODEL_BACKBONE, "input_size": INPUT_SIZE, **parameter_info, "checkpoint_size_mb": checkpoint_size_mb, "state_dict_size_mb": state_dict_size_mb}, "model_only_latency": model_latency_summary, "jetson_orin_estimation": jetson_estimation}
with open(METRIC_DIR / "edge_profiling_summary.json", "w", encoding="utf-8") as f: json.dump(profiling_summary, f, indent=4)
pd.DataFrame([{**parameter_info, **model_latency_summary, "checkpoint_size_mb": checkpoint_size_mb, "state_dict_size_mb": state_dict_size_mb, "feasibility": jetson_estimation["feasibility"], "risk_score": jetson_estimation["risk_score"]}]).to_csv(METRIC_DIR / "edge_profiling_summary_flat.csv", index=False)
pd.DataFrame({"latency_sec": latency_records, "latency_ms": latency_ms}).to_csv(METRIC_DIR / "latency_records_model_only.csv", index=False)
display(pd.DataFrame([{**model_latency_summary, "feasibility": jetson_estimation["feasibility"]}]))


## Cell 16 — Export Model ke ONNX

In [ ]:
inference_model.eval()
ONNX_DIR = OUTPUT_DIR / "onnx"
ONNX_DIR.mkdir(parents=True, exist_ok=True)
ONNX_MODEL_NAME = f"maskrcnn_mobilenetv3_fpn_{INPUT_SIZE}.onnx"
ONNX_MODEL_PATH = ONNX_DIR / ONNX_MODEL_NAME
OPSET_VERSION = 11

dummy_input = [torch.randn(3, INPUT_SIZE, INPUT_SIZE, dtype=torch.float32, device=device)]
with torch.no_grad():
    test_output = inference_model(dummy_input)
print("Forward sebelum export OK:", list(test_output[0].keys()))

torch.onnx.export(
    inference_model,
    dummy_input,
    str(ONNX_MODEL_PATH),
    input_names=["images"],
    output_names=["boxes", "labels", "scores", "masks"],
    opset_version=OPSET_VERSION,
    do_constant_folding=True,
    dynamo=False
)
summary = {"onnx_model_path": str(ONNX_MODEL_PATH), "input_size": INPUT_SIZE, "opset_version": OPSET_VERSION, "onnx_size_mb": ONNX_MODEL_PATH.stat().st_size/(1024**2)}
with open(METRIC_DIR / "onnx_export_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)
display(pd.DataFrame([summary]))


## Cell 17 — Validasi ONNX Inference di Laptop Menggunakan ONNX Runtime

In [ ]:
try:
    import onnxruntime as ort
    import onnx
except ImportError as e:
    raise ImportError("Install dahulu: pip install onnxruntime onnx") from e

if "ONNX_MODEL_PATH" not in globals():
    ONNX_MODEL_PATH = OUTPUT_DIR / "onnx" / f"maskrcnn_mobilenetv3_fpn_{INPUT_SIZE}.onnx"
onnx_model = onnx.load(str(ONNX_MODEL_PATH))
onnx.checker.check_model(onnx_model)
providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if "CUDAExecutionProvider" in ort.get_available_providers() and torch.cuda.is_available() else ["CPUExecutionProvider"]
ort_session = ort.InferenceSession(str(ONNX_MODEL_PATH), providers=providers)
ORT_INPUT_NAME = ort_session.get_inputs()[0].name
ORT_OUTPUT_NAMES = [o.name for o in ort_session.get_outputs()]
print("Provider:", ort_session.get_providers())
print("Input:", ORT_INPUT_NAME)
print("Outputs:", ORT_OUTPUT_NAMES)

if "test_images" not in globals():
    test_images = sorted(list(TEST_IMG_DIR.glob("*.jpg")) + list(TEST_IMG_DIR.glob("*.jpeg")) + list(TEST_IMG_DIR.glob("*.png")))
image = Image.open(test_images[0]).convert("RGB").resize((INPUT_SIZE, INPUT_SIZE))
x = np.expand_dims(np.transpose(np.array(image).astype(np.float32)/255.0, (2,0,1)), 0).astype(np.float32)
start = time.time()
outs = ort_session.run(ORT_OUTPUT_NAMES, {ORT_INPUT_NAME: x})
elapsed = time.time() - start
for name, out in zip(ORT_OUTPUT_NAMES, outs):
    print(name, np.array(out).shape, np.array(out).dtype)
summary = {"onnx_model_path": str(ONNX_MODEL_PATH), "provider": ort_session.get_providers()[0], "inference_time_sec": elapsed, "input_name": ORT_INPUT_NAME, "output_names": ORT_OUTPUT_NAMES}
with open(METRIC_DIR / "onnx_runtime_validation_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)
display(pd.DataFrame([summary]))


## Cell 18 — Persiapan Deployment Jetson Orin Nano

In [ ]:
import shutil, zipfile
from datetime import datetime

if "ONNX_MODEL_PATH" not in globals():
    ONNX_MODEL_PATH = OUTPUT_DIR / "onnx" / f"maskrcnn_mobilenetv3_fpn_{INPUT_SIZE}.onnx"
if "CONFIG_PATH" not in globals():
    CONFIG_PATH = PROJECT_DIR / "configs" / "config_maskrcnn_mobilenetv3_fpn.yaml"

JETSON_DEPLOY_DIR = OUTPUT_DIR / "jetson_orin_deployment"
JETSON_SCRIPT_DIR = JETSON_DEPLOY_DIR / "scripts"
for p in [JETSON_DEPLOY_DIR / "models", JETSON_DEPLOY_DIR / "configs", JETSON_SCRIPT_DIR, JETSON_DEPLOY_DIR / "videos", JETSON_DEPLOY_DIR / "outputs"]:
    p.mkdir(parents=True, exist_ok=True)

shutil.copy2(ONNX_MODEL_PATH, JETSON_DEPLOY_DIR / "models" / ONNX_MODEL_PATH.name)
shutil.copy2(CONFIG_PATH, JETSON_DEPLOY_DIR / "configs" / CONFIG_PATH.name)
video_src = VIDEO_DIR / INPUT_VIDEO_NAME if "INPUT_VIDEO_NAME" in globals() else VIDEO_DIR / "pothole_test.mp4"
if video_src.exists():
    shutil.copy2(video_src, JETSON_DEPLOY_DIR / "videos" / video_src.name)

(JETSON_DEPLOY_DIR / "requirements_jetson_orin.txt").write_text("numpy\nopencv-python\nPillow\nPyYAML\ntqdm\nonnx\nonnxruntime\npandas\ncuda-python\npycuda\n", encoding="utf-8")
(JETSON_SCRIPT_DIR / "check_jetson_environment.py").write_text("import sys, platform\nprint('Python:', sys.version)\nprint('Platform:', platform.platform())\nfor p in ['numpy','cv2','yaml','onnx','onnxruntime']:\n    try:\n        m=__import__(p); print('[OK]', p, getattr(m,'__version__','unknown'))\n    except Exception as e: print('[FAIL]', p, e)\n", encoding="utf-8")
(JETSON_DEPLOY_DIR / "run_jetson_checks.sh").write_text("#!/bin/bash\npython3 scripts/check_jetson_environment.py\n", encoding="utf-8")
manifest = {"created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "target_device": "Jetson Orin Nano", "onnx_model": ONNX_MODEL_PATH.name, "config": CONFIG_PATH.name}
with open(JETSON_DEPLOY_DIR / "deployment_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)
print("Deployment folder:", JETSON_DEPLOY_DIR)


## Cell 19 — Script Inferensi ONNX di Jetson Orin Nano

In [ ]:
script = """#!/usr/bin/env python3
import argparse, json, time
from pathlib import Path
import cv2, numpy as np, yaml, onnxruntime as ort

def load_yaml(p):
    with open(p, 'r', encoding='utf-8') as f: return yaml.safe_load(f)

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--model', default='models/model.onnx')
    ap.add_argument('--config', default='configs/config_maskrcnn_mobilenetv3_fpn.yaml')
    ap.add_argument('--source', default='videos/pothole_test.mp4')
    ap.add_argument('--output', default='outputs/onnx_video_prediction.mp4')
    ap.add_argument('--max-frames', type=int, default=100)
    args = ap.parse_args()
    cfg = load_yaml(args.config); size = int(cfg['dataset']['input_size'])
    providers = ort.get_available_providers()
    use = ['TensorrtExecutionProvider','CUDAExecutionProvider','CPUExecutionProvider'] if 'TensorrtExecutionProvider' in providers else (['CUDAExecutionProvider','CPUExecutionProvider'] if 'CUDAExecutionProvider' in providers else ['CPUExecutionProvider'])
    sess = ort.InferenceSession(args.model, providers=use)
    inp = sess.get_inputs()[0].name; outs = [o.name for o in sess.get_outputs()]
    cap = cv2.VideoCapture(int(args.source) if str(args.source).isdigit() else args.source)
    if not cap.isOpened(): raise RuntimeError('Cannot open source')
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)); fps = cap.get(cv2.CAP_PROP_FPS) or 30
    Path(args.output).parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(args.output, cv2.VideoWriter_fourcc(*'mp4v'), fps, (W,H))
    times=[]; idx=0
    while True:
        ret, frame = cap.read()
        if not ret or (args.max_frames is not None and idx >= args.max_frames): break
        rgb = cv2.cvtColor(cv2.resize(frame, (size,size)), cv2.COLOR_BGR2RGB).astype(np.float32)/255.0
        x = np.expand_dims(np.transpose(rgb,(2,0,1)),0).astype(np.float32)
        t=time.time(); y=sess.run(outs,{inp:x}); times.append(time.time()-t)
        cv2.putText(frame, f'ONNX FPS {1/np.mean(times):.2f}', (20,35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)
        writer.write(frame); idx += 1
    cap.release(); writer.release()
    summary={'frames_read':idx,'avg_model_latency_sec':float(np.mean(times)) if times else 0,'avg_model_fps':float(1/np.mean(times)) if times else 0,'providers':sess.get_providers()}
    with open(Path(args.output).parent/'onnx_video_inference_summary.json','w') as f: json.dump(summary,f,indent=4)
    print(summary)
if __name__ == '__main__': main()
"""
(JETSON_SCRIPT_DIR / "infer_onnx_video_jetson.py").write_text(script, encoding="utf-8")
(JETSON_DEPLOY_DIR / "run_onnx_video_inference.sh").write_text("#!/bin/bash\nMODEL=$(ls models/*.onnx | head -n 1)\npython3 scripts/infer_onnx_video_jetson.py --model \"$MODEL\" --config configs/config_maskrcnn_mobilenetv3_fpn.yaml --source videos/pothole_test.mp4 --output outputs/onnx_video_prediction.mp4\n", encoding="utf-8")
print("ONNX Jetson script created")


## Cell 20 — Konversi ONNX ke TensorRT FP16

In [ ]:
trt_script = """#!/bin/bash
set -e
MODEL=${1:-$(ls models/*.onnx | head -n 1)}
BASE=$(basename "$MODEL" .onnx)
TRTEXEC=$(command -v trtexec || true)
if [ -z "$TRTEXEC" ] && [ -f /usr/src/tensorrt/bin/trtexec ]; then TRTEXEC=/usr/src/tensorrt/bin/trtexec; fi
if [ -z "$TRTEXEC" ]; then echo 'trtexec not found'; exit 1; fi
mkdir -p outputs/tensorrt_logs
$TRTEXEC --onnx="$MODEL" --saveEngine="models/${BASE}_fp16.engine" --fp16 --memPoolSize=workspace:4096 --verbose 2>&1 | tee "outputs/tensorrt_logs/${BASE}_fp16_build.log" || \
$TRTEXEC --onnx="$MODEL" --saveEngine="models/${BASE}_fp32.engine" --memPoolSize=workspace:4096 --verbose 2>&1 | tee "outputs/tensorrt_logs/${BASE}_fp32_build.log"
"""
(JETSON_DEPLOY_DIR / "convert_onnx_to_tensorrt_fp16.sh").write_text(trt_script, encoding="utf-8")
(JETSON_DEPLOY_DIR / "run_tensorrt_conversion.sh").write_text("#!/bin/bash\nMODEL=$(ls models/*.onnx | head -n 1)\nchmod +x convert_onnx_to_tensorrt_fp16.sh\n./convert_onnx_to_tensorrt_fp16.sh \"$MODEL\"\n", encoding="utf-8")
(JETSON_DEPLOY_DIR / "TENSORRT_MANUAL_COMMANDS.md").write_text("# TensorRT Manual Commands\n\ntrtexec --onnx=models/model.onnx --saveEngine=models/model_fp16.engine --fp16 --memPoolSize=workspace:4096 --verbose\n", encoding="utf-8")
print("TensorRT conversion scripts created")


## Cell 21 — Script Inferensi/Benchmark TensorRT Engine

In [ ]:
script = """#!/usr/bin/env python3
import argparse, subprocess, json, time
from pathlib import Path
ap=argparse.ArgumentParser(); ap.add_argument('--engine',default=None); ap.add_argument('--output-summary',default='outputs/tensorrt_engine_benchmark_summary.json'); args=ap.parse_args()
engine = Path(args.engine) if args.engine else (next(Path('models').glob('*_fp16.engine'), None) or next(Path('models').glob('*_fp32.engine'), None))
if engine is None: raise FileNotFoundError('No engine found')
trtexec=subprocess.run(['bash','-lc','command -v trtexec || echo /usr/src/tensorrt/bin/trtexec'],capture_output=True,text=True).stdout.strip()
cmd=[trtexec, f'--loadEngine={engine}', '--warmUp=500', '--duration=10', '--iterations=100']
t=time.time(); p=subprocess.run(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True); elapsed=time.time()-t
print(p.stdout)
Path(args.output_summary).parent.mkdir(parents=True,exist_ok=True)
with open(args.output_summary,'w') as f: json.dump({'engine':str(engine),'returncode':p.returncode,'elapsed_sec':elapsed,'raw_log':p.stdout},f,indent=4)
"""
(JETSON_SCRIPT_DIR / "infer_tensorrt_video_jetson.py").write_text(script, encoding="utf-8")
(JETSON_DEPLOY_DIR / "run_tensorrt_video_inference.sh").write_text("#!/bin/bash\nENGINE=$(ls models/*_fp16.engine 2>/dev/null | head -n 1); if [ -z \"$ENGINE\" ]; then ENGINE=$(ls models/*_fp32.engine 2>/dev/null | head -n 1); fi\npython3 scripts/infer_tensorrt_video_jetson.py --engine \"$ENGINE\"\n", encoding="utf-8")
print("TensorRT benchmark script created")


## Cell 22 — Benchmark ONNX Runtime vs TensorRT

In [ ]:
script = """#!/usr/bin/env python3
import argparse, json, subprocess, sys
from pathlib import Path
import pandas as pd
ap=argparse.ArgumentParser(); ap.add_argument('--output-dir',default='outputs/benchmark_onnx_vs_tensorrt'); args=ap.parse_args()
out=Path(args.output_dir); out.mkdir(parents=True,exist_ok=True)
logs=out/'logs'; logs.mkdir(exist_ok=True)
onnx=next(Path('models').glob('*.onnx'), None)
engine=next(Path('models').glob('*_fp16.engine'), None) or next(Path('models').glob('*_fp32.engine'), None)
row={'onnx_model':str(onnx) if onnx else None,'engine':str(engine) if engine else None}
if onnx:
    subprocess.run([sys.executable,'scripts/infer_onnx_video_jetson.py','--model',str(onnx),'--config','configs/config_maskrcnn_mobilenetv3_fpn.yaml','--source','videos/pothole_test.mp4','--output',str(out/'onnx_runtime_prediction.mp4')])
    p=out/'onnx_video_inference_summary.json'
    if p.exists(): row.update(json.load(open(p)))
if engine:
    subprocess.run([sys.executable,'scripts/infer_tensorrt_video_jetson.py','--engine',str(engine),'--output-summary',str(out/'tensorrt_engine_benchmark_summary.json')])
row['tensorrt_engine_available']=bool(engine)
with open(out/'benchmark_onnx_vs_tensorrt_summary.json','w') as f: json.dump(row,f,indent=4)
pd.DataFrame([row]).to_csv(out/'benchmark_onnx_vs_tensorrt_summary.csv',index=False)
(out/'benchmark_onnx_vs_tensorrt_report.md').write_text('# Benchmark Report\\n\\n```json\\n'+json.dumps(row,indent=2)+'\\n```\\n')
print(row)
"""
(JETSON_SCRIPT_DIR / "benchmark_onnx_vs_tensorrt_jetson.py").write_text(script, encoding="utf-8")
(JETSON_DEPLOY_DIR / "run_benchmark_onnx_vs_tensorrt.sh").write_text("#!/bin/bash\npython3 scripts/benchmark_onnx_vs_tensorrt_jetson.py\n", encoding="utf-8")
print("Benchmark script created")


## Cell 23 — Generator Laporan Akhir Deployment Jetson Orin Nano

In [ ]:
script = """#!/usr/bin/env python3
import json
from pathlib import Path
from datetime import datetime
import pandas as pd, yaml
out=Path('outputs/final_deployment_report'); out.mkdir(parents=True,exist_ok=True)
cfg_path=next(Path('configs').glob('*.yaml'), None)
cfg=yaml.safe_load(open(cfg_path)) if cfg_path else {}
bench_path=Path('outputs/benchmark_onnx_vs_tensorrt/benchmark_onnx_vs_tensorrt_summary.json')
bench=json.load(open(bench_path)) if bench_path.exists() else {}
summary={'created_at':datetime.now().isoformat(timespec='seconds'),'target_device':'Jetson Orin Nano','architecture':cfg.get('model',{}).get('architecture','maskrcnn'),'backbone':cfg.get('model',{}).get('backbone','mobilenetv3_large_fpn'),**bench}
with open(out/'jetson_orin_deployment_summary.json','w') as f: json.dump(summary,f,indent=4)
pd.DataFrame([summary]).to_csv(out/'jetson_orin_deployment_summary.csv',index=False)
md=['# Final Jetson Orin Nano Deployment Report','']+[f'- **{k}**: {v}' for k,v in summary.items()]
(out/'jetson_orin_deployment_report.md').write_text('\\n'.join(md))
print('Final report saved:', out)
"""
(JETSON_SCRIPT_DIR / "generate_deployment_report_jetson.py").write_text(script, encoding="utf-8")
(JETSON_DEPLOY_DIR / "run_generate_deployment_report.sh").write_text("#!/bin/bash\npython3 scripts/generate_deployment_report_jetson.py\n", encoding="utf-8")
for sh in JETSON_DEPLOY_DIR.glob("*.sh"): sh.chmod(0o755)
for py in JETSON_SCRIPT_DIR.glob("*.py"): py.chmod(0o755)
JETSON_DEPLOY_ZIP = OUTPUT_DIR / f"jetson_orin_deployment_{INPUT_SIZE}_final.zip"
if JETSON_DEPLOY_ZIP.exists(): JETSON_DEPLOY_ZIP.unlink()
with zipfile.ZipFile(JETSON_DEPLOY_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for fp in JETSON_DEPLOY_DIR.rglob("*"):
        if fp.is_file(): zf.write(fp, fp.relative_to(JETSON_DEPLOY_DIR.parent))
print("Final deployment ZIP:", JETSON_DEPLOY_ZIP)
